# Economics Science — R Data Structures
## Solution Notebook

**Sector:** Economics Science / Empirical Macro & Development  
**Goal:** Master vectors, matrices, lists, data frames and factors by building research-grade economic objects.

![Flowchart](economics_science_data_structures_flowchart.png)


## 0. Setup


In [ ]:
library(dplyr)

panel <- read.csv("data/economics_panel.csv", stringsAsFactors = FALSE)
str(panel)
cat("Countries:", length(unique(panel$country)),
    " | Years:", length(unique(panel$year)), "\n")


## 1. Vectors — Atomic Time Series


In [ ]:
# USA GDP growth vector
usa_growth <- panel$gdp_growth[panel$country == "USA"]
names(usa_growth) <- panel$year[panel$country == "USA"]
print(usa_growth)

cat("Mean:", mean(usa_growth),
    " Var:", var(usa_growth),
    " SD:", sd(usa_growth), "\n")

# Named vector of average inflation by year
yearly_inf <- tapply(panel$inflation, panel$year, mean)
print(round(yearly_inf, 2))


## 2. Matrices — Relationships


In [ ]:
nums <- panel[, c("gdp_growth", "inflation", "unemployment", "trade_openness")]
cor_mat <- cor(nums)
cov_mat <- cov(nums)

print(round(cor_mat, 3))
cat("\nGDP growth vs Inflation correlation:",
    round(cor_mat["gdp_growth", "inflation"], 3), "\n")

print(round(cov_mat, 2))


## 3. Lists — Country Objects


In [ ]:
make_country_obj <- function(df, ctry) {
  sub <- df[df$country == ctry, ]
  list(
    name         = ctry,
    region       = unique(sub$region),
    income_group = unique(sub$income_group),
    growth       = setNames(sub$gdp_growth, sub$year),
    inflation    = setNames(sub$inflation, sub$year)
  )
}

usa_obj     <- make_country_obj(panel, "USA")
germany_obj <- make_country_obj(panel, "Germany")
india_obj   <- make_country_obj(panel, "India")

countries_list <- list(USA = usa_obj, Germany = germany_obj, India = india_obj)

str(countries_list, max.level = 2)

# Two ways to extract nested data
cat("India growth via [[ ]]:\n")
print(countries_list[["India"]][["growth"]])
cat("India growth via $ :\n")
print(countries_list$India$growth)


## 4. Data Frames & Factors


In [ ]:
panel$region <- factor(panel$region)
panel$income_group <- factor(panel$income_group,
                             levels = c("Lower-middle", "Upper-middle", "High"),
                             ordered = TRUE)

cat("Region levels:", levels(panel$region), "\n")
cat("Income levels (ordered):", levels(panel$income_group), "\n")

region_summary <- panel %>%
  group_by(region) %>%
  summarise(
    mean_growth     = round(mean(gdp_growth), 2),
    mean_inflation  = round(mean(inflation), 2),
    mean_unemp      = round(mean(unemployment), 2),
    n_obs           = n()
  )
print(region_summary)


## 5. Alternate Code Paths (base R)


In [ ]:
# aggregate
agg1 <- aggregate(cbind(gdp_growth, inflation, unemployment) ~ region,
                  data = panel, FUN = mean)
print(round(agg1, 2))

# tapply style
growth_by_reg <- tapply(panel$gdp_growth, panel$region, mean)
print(round(growth_by_reg, 2))

# split + lapply
by_reg <- split(panel, panel$region)
means_list <- lapply(by_reg, function(d) colMeans(d[, c("gdp_growth","inflation","unemployment")]))
print(lapply(means_list, round, 2))


## 6. More Practice


In [ ]:
# Highest average unemployment by income group
unemp_by_inc <- tapply(panel$unemployment, panel$income_group, mean)
print(round(unemp_by_inc, 2))
cat("Highest unemployment income group:", names(which.max(unemp_by_inc)), "\n")

# Country × year matrix of GDP growth
growth_mat <- tapply(panel$gdp_growth, list(panel$country, panel$year), identity)
print(growth_mat[1:5, 1:6])   # preview

# Enrich a country object with its own indicator correlation
usa_nums <- panel[panel$country == "USA", c("gdp_growth","inflation","unemployment","trade_openness")]
usa_obj$cor <- cor(usa_nums)
print(round(usa_obj$cor, 3))


## 7. Simulation — Volatility & Sampling Precision


In [ ]:
set.seed(42)
n_years <- 11
n_sims  <- 500
vols    <- seq(0.5, 3.0, by = 0.5)

sim_results <- lapply(vols, function(v) {
  means <- replicate(n_sims, mean(rnorm(n_years, mean = 3.0, sd = v)))
  c(avg_of_means = mean(means), sd_of_means = sd(means))
})
sim_df <- data.frame(volatility = vols, do.call(rbind, sim_results))
print(round(sim_df, 3))

par(mfrow = c(1, 2))
plot(sim_df$volatility, sim_df$avg_of_means, type = "b", pch = 19, col = "darkgreen",
     main = "Mean of simulated growth", xlab = "Volatility", ylab = "Average mean")
plot(sim_df$volatility, sim_df$sd_of_means, type = "b", pch = 19, col = "darkred",
     main = "Sampling SD of the mean", xlab = "Volatility", ylab = "SD of means")


## 8. Audience-Aware Take-aways

**Economist / researcher version**  
The panel is stored as a data frame with ordered factors for income group and unordered factors for region. Correlation and covariance matrices give a quick view of linear associations among the four core indicators; the modest negative correlation between growth and inflation is visible at a glance. Nested lists keep country-specific series and metadata together, which is convenient when writing functions that operate on one economy at a time. Monte-Carlo results confirm that the standard error of a sample mean scales linearly with the volatility of the underlying process — useful when designing the length of a study or the precision targets of a forecast evaluation.

**Policy / non-specialist version**  
When we organise the numbers by country and year we can see clear regional patterns: Oceania and Asia posted the strongest average growth, while North America experienced higher average inflation. Grouping countries by income level also shows differences in labour-market outcomes. Keeping the data in a tidy table (and using simple summary tables) makes these comparisons fast and transparent for decision makers.
